In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medical_pipeline.gold;

In [0]:
%sql
-- Dimension Table: Patient Information
-- Type: SCD Type 2 (tracks historical changes)
CREATE OR REPLACE TABLE medical_pipeline.gold.dim_patient (
  patient_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  patient_id STRING NOT NULL,                        -- Natural key from source
  first_name STRING,
  last_name STRING,
  gender STRING,
  birth_date DATE,
  race STRING,
  ethnicity STRING,
  city STRING,
  state STRING,
  county STRING,
  zip STRING,
  
  -- SCD Type 2 columns
  effective_date DATE NOT NULL,
  end_date DATE,
  is_current BOOLEAN NOT NULL DEFAULT TRUE,
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_dim_patient PRIMARY KEY (patient_key)
) USING DELTA
COMMENT 'Patient dimension table with SCD Type 2 for tracking historical changes';

In [0]:
%sql
-- Dimension Table: Procedure Types
-- Type: SCD Type 1 (overwrite changes)
CREATE OR REPLACE TABLE medical_pipeline.gold.dim_procedure_type (
  procedure_type_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  procedure_code DOUBLE NOT NULL,                          -- Natural key
  procedure_description STRING NOT NULL,
  procedure_category STRING,                                -- Grouping/category
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_dim_procedure_type PRIMARY KEY (procedure_type_key),
  CONSTRAINT uk_procedure_code UNIQUE (procedure_code)
) USING DELTA
COMMENT 'Procedure type dimension - lookup for procedure codes and descriptions';

In [0]:
%sql
-- Dimension Table: Encounter Types
-- Type: SCD Type 1 (overwrite changes)
CREATE OR REPLACE TABLE medical_pipeline.gold.dim_encounter_type (
  encounter_type_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  encounter_code BIGINT NOT NULL,                          -- Natural key
  encounter_description STRING NOT NULL,
  encounter_class STRING,                                   -- ambulatory, emergency, inpatient, etc.
  encounter_category STRING,                                -- Grouping/category
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_dim_encounter_type PRIMARY KEY (encounter_type_key),
  CONSTRAINT uk_encounter_code UNIQUE (encounter_code)
) USING DELTA
COMMENT 'Encounter type dimension - lookup for encounter codes, descriptions, and classes';

In [0]:
%sql
-- Dimension Table: Insurance Payers
-- Type: SCD Type 2 (tracks historical changes)
CREATE OR REPLACE TABLE medical_pipeline.gold.dim_payer (
  payer_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  payer_id STRING NOT NULL,                       -- Natural key from source
  payer_name STRING,
  payer_type STRING,                              -- Medicare, Medicaid, Commercial, etc.
  payer_address STRING,
  payer_city STRING,
  payer_state STRING,
  payer_zip STRING,
  
  -- SCD Type 2 columns
  effective_date DATE NOT NULL,
  end_date DATE,
  is_current BOOLEAN NOT NULL DEFAULT TRUE,
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_dim_payer PRIMARY KEY (payer_key)
) USING DELTA
COMMENT 'Payer (insurance) dimension with SCD Type 2 for tracking historical changes';

In [0]:
%sql
-- Dimension Table: Reason Codes
-- Type: SCD Type 1 (overwrite changes)
CREATE OR REPLACE TABLE medical_pipeline.gold.dim_reason (
  reason_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  reason_code DOUBLE NOT NULL,                     -- Natural key
  reason_description STRING NOT NULL,
  reason_category STRING,                           -- Grouping/category
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_dim_reason PRIMARY KEY (reason_key),
  CONSTRAINT uk_reason_code UNIQUE (reason_code)
) USING DELTA
COMMENT 'Reason code dimension - lookup for medical reason codes and descriptions';

In [0]:
%sql
-- Dimension Table: Date Dimension
-- Type: Static reference table
CREATE OR REPLACE TABLE medical_pipeline.gold.dim_date (
  date_key INT NOT NULL,                     -- YYYYMMDD format (e.g., 20260326)
  full_date DATE NOT NULL,
  day_of_week INT,                           -- 1=Sunday, 7=Saturday
  day_name STRING,                           -- Monday, Tuesday, etc.
  day_of_month INT,
  day_of_year INT,
  week_of_year INT,
  month INT,
  month_name STRING,
  quarter INT,
  year INT,
  is_weekend BOOLEAN,
  is_holiday BOOLEAN,
  fiscal_year INT,
  fiscal_quarter INT,
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
) USING DELTA
COMMENT 'Date dimension for time-based analysis and reporting';

In [0]:
%sql
-- Fact Table: Procedures
-- Grain: One row per procedure performed
CREATE OR REPLACE TABLE medical_pipeline.gold.fact_procedures (
  procedure_fact_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  
  -- Foreign keys to dimensions
  patient_key BIGINT NOT NULL,
  procedure_type_key BIGINT NOT NULL,
  encounter_key BIGINT NOT NULL,
  reason_key BIGINT,
  start_date_key INT NOT NULL,
  stop_date_key INT,
  
  -- Degenerate dimensions (transaction identifiers kept in fact)
  encounter_id STRING NOT NULL,
  
  -- Date/Time stamps (for precise querying)
  start_timestamp TIMESTAMP NOT NULL,
  stop_timestamp TIMESTAMP,
  
  -- Measures (numeric facts)
  base_cost DOUBLE,
  duration_minutes DOUBLE,
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_fact_procedures PRIMARY KEY (procedure_fact_key)
) USING DELTA
PARTITIONED BY (start_date_key)
COMMENT 'Fact table for procedure events with measures and dimensional references';

In [0]:
%sql
-- Fact Table: Encounters
-- Grain: One row per patient encounter
CREATE OR REPLACE TABLE medical_pipeline.gold.fact_encounters (
  encounter_fact_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  
  -- Foreign keys to dimensions
  patient_key BIGINT NOT NULL,
  encounter_type_key BIGINT NOT NULL,
  payer_key BIGINT NOT NULL,
  reason_key BIGINT,
  start_date_key INT NOT NULL,
  stop_date_key INT,
  
  -- Degenerate dimensions (transaction identifiers kept in fact)
  encounter_id STRING NOT NULL,
  
  -- Date/Time stamps (for precise querying)
  start_timestamp TIMESTAMP NOT NULL,
  stop_timestamp TIMESTAMP,
  
  -- Measures (numeric facts)
  base_encounter_cost DOUBLE,
  total_claim_cost DOUBLE,
  payer_coverage DOUBLE,
  patient_responsibility DOUBLE,                           -- Derived: total - coverage
  duration_hours DOUBLE,                                    -- Derived: stop - start
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_fact_encounters PRIMARY KEY (encounter_fact_key),
  CONSTRAINT uk_encounter_id UNIQUE (encounter_id)
) USING DELTA
PARTITIONED BY (start_date_key)
COMMENT 'Fact table for encounter events with cost measures and dimensional references';